# So sánh kết quả giữa hai thí nghiệm Fit–Validation

Hai thí nghiệm sử dụng hai bộ tham số khác nhau nhằm so sánh hiệu quả huấn luyện và đánh giá mô hình trên tập validation.

## Nguồn thí nghiệm

1. **Params1 – Bộ tham số Claude tái dựng:**
   [20-11-2026_Huy_C30_Params1](../../experiments/20-11-2026_Huy_C30_Params1/fit_validation_C30.ipynb)

2. **Params2 – Bộ tham số hiện tại của dự án:**
   [20-11-2026_Huy_C30_Params2](../../experiments/20-11-2026_Huy_C30_Params2/fit_validation_C30.ipynb)

## Mục tiêu so sánh

- Đối chiếu cấu hình tham số giữa hai thí nghiệm.
- So sánh các chỉ số đánh giá trên tập validation.
- Phân tích mức độ chênh lệch về hiệu suất mô hình.
- Xác định bộ tham số phù hợp hơn dựa trên kết quả thực nghiệm.

## 1. Import lại output thí nghiệm

In [1]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "data/preprocessed/hourly_unfilled.csv").is_file()
)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.evaluation import regression_metrics

sources = {
    "Hiện tại": project_root
    / "experiments/20-11-2026_Huy_C30_Params2/outputs",

    "Claude": project_root
    / "experiments/20-11-2026_Huy_C30_Params1/outputs",
}

## 2. Đọc và kiểm tra hai lượt có cùng dữ liệu xét

Chỉ so sánh trực tiếp khi hai lượt dùng cùng đặc trưng, cohort train, tiền xử lý, timestamp validation và giá trị thực tế. Không tự lấy giao của hai tập giờ vì có thể che mất sự khác nhau giữa các lượt.

In [2]:
predictions = {}
configs = {}

for name, folder in sources.items():
    predictions[name] = (
        pd.read_csv(
            folder / "validation_predictions.csv",
            parse_dates=["date_time"],
        )
        .sort_values("date_time")
        .reset_index(drop=True)
    )

    configs[name] = json.loads(
        (folder / "run_config.json").read_text(encoding="utf-8")
    )

    assert predictions[name]["date_time"].is_unique
    assert predictions[name]["date_time"].notna().all()

# Cùng timestamp và giá trị mục tiêu.
pd.testing.assert_frame_equal(
    predictions["Hiện tại"][["date_time", "y_true"]],
    predictions["Claude"][["date_time", "y_true"]],
)

# Cùng dữ liệu và cách xử lý.
for key in (
    "features",
    "input_prefix_hash",
    "train_cohort_hash",
    "validation_cohort_hash",
    "preprocessing",
):
    assert configs["Hiện tại"][key] == configs["Claude"][key], key

display(pd.DataFrame([
    {
        "bộ tham số": name,
        "n_train": configs[name]["n_train"],
        "n_validation": len(frame),
        "bắt đầu validation": frame["date_time"].min(),
        "kết thúc validation": frame["date_time"].max(),
    }
    for name, frame in predictions.items()
]))

,bộ tham số,n_train,n_validation,bắt đầu validation,kết thúc validation
0,Hiện tại,22503,6050,2017-05-10 05:00:00,2018-01-19 14:00:00
1,Claude,22503,6050,2017-05-10 05:00:00,2018-01-19 14:00:00


## 3. Tính lại chỉ số từ dự đoán
Tính lại MAE, RMSE và R² từ dự đoán để kiểm tra bảng điểm đã lưu. Mọi mô hình được chấm trên cùng mục tiêu; MAE và RMSE có đơn vị xe/giờ.

In [4]:
model_names = [
    "Random Forest",
    "XGBoost",
    "Ensemble RF + XGBoost (50/50)",
    "Baseline thứ–giờ",
    "Baseline seasonal naive",
]

rows = []

for parameter_set, frame in predictions.items():
    saved_metrics = pd.read_csv(
        sources[parameter_set] / "validation_metrics.csv"
    ).set_index("model")

    for model_name in model_names:
        assert np.isfinite(frame[model_name]).all()

        scores = regression_metrics(
            frame["y_true"],
            frame[model_name],
        )

        # Điểm tính lại phải khớp điểm đã lưu.
        for metric, value in scores.items():
            assert np.isclose(
                value,
                saved_metrics.loc[model_name, metric],
                rtol=0,
                atol=1e-6,
            )

        rows.append({
            "parameter_set": parameter_set,
            "model": model_name,
            "n": len(frame),
            **scores,
        })

metrics = pd.DataFrame(rows)

display(
    metrics.sort_values(["model", "MAE"])
    .reset_index(drop=True)
    .round(3)
)

,parameter_set,model,n,MAE,RMSE,R2
0,Hiện tại,Baseline seasonal naive,6050,330.178,635.073,0.895
1,Claude,Baseline seasonal naive,6050,330.178,635.073,0.895
2,Hiện tại,Baseline thứ–giờ,6050,291.739,503.219,0.934
3,Claude,Baseline thứ–giờ,6050,291.739,503.219,0.934
4,Claude,Ensemble RF + XGBoost (50/50),6050,193.810,316.085,0.974
5,Hiện tại,Ensemble RF + XGBoost (50/50),6050,197.229,322.497,0.973
6,Claude,Random Forest,6050,196.775,326.773,0.972
7,Hiện tại,Random Forest,6050,200.044,334.531,0.971
8,Claude,XGBoost,6050,197.031,313.640,0.975
9,Hiện tại,XGBoost,6050,201.846,322.441,0.973


## 4. So sánh bộ Claude với bộ hiện tại cho từng mô hình
Giữ cùng mô hình rồi so hai bộ tham số. Quy ước $ΔMAE = MAE_{hiện tại} − MAE_{Claude}$; $ΔRMSE$ tương tự. Giá trị dương nghĩa bộ Claude có sai số thấp hơn.

In [5]:
current = (
    metrics[metrics["parameter_set"] == "Hiện tại"]
    .set_index("model")
)

claude = (
    metrics[metrics["parameter_set"] == "Claude"]
    .set_index("model")
)

comparison = pd.DataFrame({
    "n": current["n"],
    "MAE hiện tại": current["MAE"],
    "MAE Claude": claude["MAE"],
    "ΔMAE": current["MAE"] - claude["MAE"],
    "RMSE hiện tại": current["RMSE"],
    "RMSE Claude": claude["RMSE"],
    "ΔRMSE": current["RMSE"] - claude["RMSE"],
}).reset_index()

display(comparison.round(3))

,model,n,MAE hiện tại,MAE Claude,ΔMAE,RMSE hiện tại,RMSE Claude,ΔRMSE
0,Random Forest,6050,200.044,196.775,3.269,334.531,326.773,7.758
1,XGBoost,6050,201.846,197.031,4.815,322.441,313.640,8.801
2,Ensemble RF + XGBoost (50/50),6050,197.229,193.810,3.419,322.497,316.085,6.413
3,Baseline thứ–giờ,6050,291.739,291.739,0.000,503.219,503.219,0.000
4,Baseline seasonal naive,6050,330.178,330.178,0.000,635.073,635.073,0.000


## 5. Xếp hạng ứng viên và viết nhận xét
Xếp hạng theo MAE trên validation. Kiểm tra riêng phương án có RMSE thấp nhất để nhận ra sự đánh đổi giữa hai tiêu chí. Đây là kết quả trên validation đã được xem nhiều lần, chưa phải kết luận từ test.

In [7]:
candidates = metrics[
    ~metrics["model"].str.startswith("Baseline")
].copy()

ranking = candidates.sort_values(
    ["MAE", "RMSE"]
).reset_index(drop=True)

display(ranking.round(3))

best_mae = ranking.iloc[0]
best_rmse = candidates.loc[candidates["RMSE"].idxmin()]

output_dir = project_root / "notebooks/validation/outputs"
output_dir.mkdir(parents=True, exist_ok=True)

metrics.to_csv(output_dir / "validation_metrics_all.csv", index=False)
comparison.to_csv(output_dir / "parameter_comparison.csv", index=False)
ranking.to_csv(output_dir / "validation_ranking.csv", index=False)

,parameter_set,model,n,MAE,RMSE,R2
0,Claude,Ensemble RF + XGBoost (50/50),6050,193.810,316.085,0.974
1,Claude,Random Forest,6050,196.775,326.773,0.972
2,Claude,XGBoost,6050,197.031,313.640,0.975
3,Hiện tại,Ensemble RF + XGBoost (50/50),6050,197.229,322.497,0.973
4,Hiện tại,Random Forest,6050,200.044,334.531,0.971
5,Hiện tại,XGBoost,6050,201.846,322.441,0.973


Trên cùng 6,050 giờ validation, Ensemble RF + XGBoost (50/50) — bộ Claude có MAE thấp nhất: 193.810 xe/giờ. Phương án có RMSE thấp nhất là XGBoost — bộ Claude, đạt 313.640 xe/giờ.

Nếu giữ MAE là tiêu chí chính đã quy định, phương án đứng đầu MAE là ứng viên để chốt. Chênh lệch này chưa tự chứng minh cải thiện có ý nghĩa thống kê hoặc sẽ giữ nguyên trên test.

## Quyết định cuối ngày 22/11/2026

Kết quả và kết luận phía trên thuộc phạm vi thí nghiệm lịch sử của notebook này. Nhóm đã chốt ensemble C30 ba mô hình: RF 25%, XGBoost 25%, ExtraTrees 50%. Xem [validation_final_C30.ipynb](validation_final_C30.ipynb) và [đặc tả đã khóa](outputs/final_C30/selected_model.json) cho bước fit cuối. Chưa refit trên train + validation và chưa chấm test.
